# 🤖 ShadBotTrader — Google Colab Training

**قبل از اجرا — فقط یه کار:**
- از منوی بالا: `Runtime → Change runtime type → T4 GPU`

بقیه همه چیز خودکاره.

---

## ✅ مرحله ۱ — بررسی GPU

In [ ]:
import subprocess

result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                        capture_output=True, text=True)
if result.returncode == 0:
    gpu_info = result.stdout.strip()
    print(f'✅ GPU: {gpu_info}')
else:
    print('❌ GPU پیدا نشد!')
    print('   Runtime → Change runtime type → T4 GPU')
    raise SystemExit('GPU required')

## 📦 مرحله ۲ — clone پروژه از GitHub

In [ ]:
import os, sys

REPO_URL  = 'https://github.com/DeadBotKing/ShadBotTrader'
WORK_DIR  = '/content/ShadBotTrader'

if os.path.exists(WORK_DIR):
    print('🔄 repo موجوده — pull آخرین تغییرات ...')
    !cd {WORK_DIR} && git pull
else:
    print(f'📥 clone از GitHub ...')
    !git clone {REPO_URL} {WORK_DIR}

# اضافه کردن src به path
SRC = f'{WORK_DIR}/src'
for p in [WORK_DIR, SRC]:
    if p not in sys.path:
        sys.path.insert(0, p)

os.chdir(WORK_DIR)
print(f'\n✅ پروژه آماده: {WORK_DIR}')
print(f'📁 محتوا: {[x for x in os.listdir(WORK_DIR) if not x.startswith(".")]}')

## 📦 مرحله ۳ — نصب dependencies

In [ ]:
# نصب همه dependencies
print('📦 نصب packages ...')
!pip install -q tensorflow PyYAML numpy pandas pyarrow PyWavelets

# تأیید TensorFlow + GPU
import tensorflow as tf
print(f'\n✅ TensorFlow {tf.__version__}')

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print(f'✅ GPU فعال: {[g.name for g in gpus]}')
else:
    print('⚠️  GPU شناسایی نشد — CPU استفاده میشه')

# تست import پروژه
try:
    from ShadBotTrader.infrastructure.ai.model_roles import range_model_role, signal_model_role
    role_r = range_model_role('1H')
    role_s = signal_model_role('5M')
    print(f'✅ پروژه import شد (range loss={role_r.loss}, signal loss={role_s.loss})')
except Exception as e:
    print(f'❌ خطا در import: {e}')

## 💾 مرحله ۴ — اتصال Google Drive (دیتاست)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('✅ Google Drive متصل شد')

In [ ]:
import os, zipfile, shutil

# ─────────────────────────────────────────────────────────────────────────
# 📌 اینجا رو تنظیم کن:
#    مسیر فایل datasets.zip یا پوشه datasets در Google Drive
# ─────────────────────────────────────────────────────────────────────────

DRIVE_DATASETS_PATH = '/content/drive/MyDrive/ShadBot/datasets.zip'
# یا اگه پوشه datasets مستقیم توی Drive هست:
# DRIVE_DATASETS_PATH = '/content/drive/MyDrive/ShadBot/datasets'

# ─────────────────────────────────────────────────────────────────────────

DATASETS_DIR = f'{WORK_DIR}/datasets'

if not os.path.exists(DRIVE_DATASETS_PATH):
    print(f'❌ مسیر پیدا نشد: {DRIVE_DATASETS_PATH}')
    print('   DRIVE_DATASETS_PATH رو تنظیم کن')
elif DRIVE_DATASETS_PATH.endswith('.zip'):
    print(f'📦 Extract دیتاست از {DRIVE_DATASETS_PATH} ...')
    with zipfile.ZipFile(DRIVE_DATASETS_PATH, 'r') as zf:
        zf.extractall(WORK_DIR)
    print(f'✅ Extract شد به {DATASETS_DIR}')
else:
    # پوشه مستقیم — symlink برای سرعت
    if os.path.exists(DATASETS_DIR):
        shutil.rmtree(DATASETS_DIR)
    os.symlink(DRIVE_DATASETS_PATH, DATASETS_DIR)
    print(f'✅ symlink ساخته شد: {DATASETS_DIR} → {DRIVE_DATASETS_PATH}')

# نمایش تایم‌فریم‌های موجود
processed = f'{DATASETS_DIR}/processed'
if os.path.exists(processed):
    print('\n📈 دیتاست‌های موجود:')
    for symbol in os.listdir(processed):
        sym_path = f'{processed}/{symbol}'
        if os.path.isdir(sym_path):
            tfs = sorted(os.listdir(sym_path))
            print(f'  {symbol}: {tfs}')
else:
    print('⚠️  پوشه processed پیدا نشد')

## ⚙️ مرحله ۵ — تنظیمات آموزش

**فقط این سلول رو ویرایش کن:**

In [ ]:
# ════════════════════════════════════════════════
#  تنظیمات آموزش — اینجا رو ویرایش کن
# ════════════════════════════════════════════════

CONFIG = dict(
    symbol           = 'XAUUSD',

    # ── کدوم مدل؟ ─────────────────────────────
    # 'range'  → مدل رنج
    # 'signal' → مدل سیگنال
    # 'all'    → هر دو
    model            = 'range',

    # ── تایم‌فریم ──────────────────────────────
    range_timeframes = '1H',   # '1H' یا '1D' یا '1H,1D'
    signal_timeframe = '5M',

    # ── هایپرپارامترها ─────────────────────────
    epochs           = 30,
    folds            = 3,
    window           = 150,
    train_ratio      = 80.0,
    learning_rate    = 3e-5,
    threshold        = 0.006,   # فقط برای signal
)

# ════════════════════════════════════════════════

print('✅ تنظیمات آموزش:')
for k, v in CONFIG.items():
    print(f'  {k:20s}: {v}')

## 🚀 مرحله ۶ — اجرای آموزش

In [ ]:
import subprocess, sys, os, time

cmd = [
    sys.executable,
    f'{WORK_DIR}/scripts/run_dual_models.py',
    '--with-features',
    '--symbol',           CONFIG['symbol'],
    '--model',            CONFIG['model'],
    '--signal-timeframe', CONFIG['signal_timeframe'],
    '--range-timeframes', CONFIG['range_timeframes'],
    '--epochs',           str(CONFIG['epochs']),
    '--folds',            str(CONFIG['folds']),
    '--window',           str(CONFIG['window']),
    '--train-ratio',      str(CONFIG['train_ratio']),
    '--learning-rate',    str(CONFIG['learning_rate']),
    '--threshold',        str(CONFIG['threshold']),
    '--storage-root',     DATASETS_DIR,
]

env = {**os.environ,
       'TF_CPP_MIN_LOG_LEVEL': '2',
       'PYTHONPATH': f'{WORK_DIR}/src'}

print('🚀 شروع آموزش ...')
print('─' * 60)
t0 = time.time()

process = subprocess.Popen(
    cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1, cwd=WORK_DIR, env=env
)

for line in process.stdout:
    print(line, end='', flush=True)

process.wait()
elapsed = time.time() - t0
mins, secs = divmod(int(elapsed), 60)
hours, mins = divmod(mins, 60)

print('─' * 60)
if process.returncode == 0:
    print(f'\n✅ آموزش تموم شد — {hours:02d}:{mins:02d}:{secs:02d}')
else:
    print(f'\n❌ خطا (code={process.returncode})')

## 💾 مرحله ۷ — ذخیره مدل روی Google Drive

In [ ]:
import os, zipfile, shutil

# ─────────────────────────────────────────────────────────────────────────
# 📌 مسیر ذخیره در Drive (میتونی تغییر بدی)
DRIVE_SAVE_PATH = '/content/drive/MyDrive/ShadBot/trained_models'
# ─────────────────────────────────────────────────────────────────────────

MODELS_DIR = f'{DATASETS_DIR}/models'

if not os.path.exists(MODELS_DIR):
    print('❌ پوشه models پیدا نشد — آموزش را ابتدا اجرا کن')
else:
    # نمایش مدل‌های ذخیره‌شده
    print('📁 مدل‌های ذخیره‌شده:')
    for model_name in sorted(os.listdir(MODELS_DIR)):
        model_path = f'{MODELS_DIR}/{model_name}'
        if os.path.isdir(model_path):
            files_list = sorted(os.listdir(model_path))
            size_mb = sum(
                os.path.getsize(f'{model_path}/{f}')
                for f in files_list
            ) / 1024 / 1024
            print(f'  📦 {model_name}: {files_list}  ({size_mb:.1f} MB)')

    # کپی به Drive
    os.makedirs(DRIVE_SAVE_PATH, exist_ok=True)
    print(f'\n📤 کپی به Drive: {DRIVE_SAVE_PATH} ...')

    for model_name in os.listdir(MODELS_DIR):
        src = f'{MODELS_DIR}/{model_name}'
        dst = f'{DRIVE_SAVE_PATH}/{model_name}'
        if os.path.isdir(src):
            if os.path.exists(dst):
                shutil.rmtree(dst)
            shutil.copytree(src, dst)
            print(f'  ✅ {model_name} → Drive')

    print(f'\n✅ همه مدل‌ها روی Drive ذخیره شدن: {DRIVE_SAVE_PATH}')

## 🔄 آموزش مدل دیگه (اختیاری)

برای آموزش مدل signal، مرحله ۵ رو تغییر بده و مرحله ۶ رو دوباره اجرا کن:

In [ ]:
# تنظیمات سریع برای مدل signal:
CONFIG = dict(
    symbol           = 'XAUUSD',
    model            = 'signal',
    range_timeframes = '1H',
    signal_timeframe = '5M',
    epochs           = 30,
    folds            = 3,
    window           = 150,
    train_ratio      = 80.0,
    learning_rate    = 1e-4,
    threshold        = 0.006,
)

print('✅ CONFIG برای signal آماده — مرحله ۶ رو دوباره اجرا کن')

---

## 📝 راهنمای سریع

### مسیر datasets در Drive:
```
MyDrive/
  ShadBot/
    datasets.zip   ← یا پوشه datasets
```

### هایپرپارامترها:
| پارامتر | Range (1H) | Signal (5M) |
|---------|-----------|-------------|
| `epochs` | 30-50 | 30-50 |
| `folds` | 3 | 3 |
| `window` | 150 | 150 |
| `learning_rate` | 3e-5 | 1e-4 |
| `threshold` | - | 0.006 |

### سرعت با T4 GPU:
| | CPU | T4 GPU |
|--|-----|--------|
| هر epoch | ~13 min | **~1-2 min** |
| 30 epoch × 3 fold | ~11 ساعت | **~1 ساعت** |

### بعد از آموزش:
مدل‌ها روی Drive ذخیره میشن → پوشه `trained_models/` رو دانلود کن و داخل `datasets/models/` سیستمت کپی کن.